# Building-классификатор: ConvNeXt-Tiny (3 класса, Zenodo NY small)

Аналог `03_convnext_training.ipynb`, но для классификации отдельных **зданий**
(не зон): `residential` / `commercial` / `industrial` по crop'ам зданий из
Zenodo Building Type small (`data/processed_ny_building/`).

Маппинг: Single+Multi → `residential`, Commercial/Industrial отдельно. High/Hospital/Schools не используются.
Дисбаланс умеренный (~422 / ~202 / ~214), но выборка маленькая (~838 crop'ов, Нью-Йорк).

Улучшения против переобучения / minority-классов:
- `WeightedRandomSampler` + **Focal loss** + label smoothing
- early stop по **val macro F1**; лимит эпох на подэтап stage2
- mask-guided фон (серый вне здания) из `*_mask.png`; маска ≈ квадрат по `pixle_per_side` из CSV Zenodo

Результат: `models/ny_building_classifier.pth` — используется в
`notebooks/08_zone_building_pipeline.ipynb` (после 06 fine-tune и 07 калибровки — опционально).


In [ ]:
import os
from pathlib import Path

# Токен Hugging Face — из .env в корне проекта (не коммитится в git)
_env_path = Path('..').resolve() / '.env'
if _env_path.exists():
    for _line in _env_path.read_text(encoding='utf-8').splitlines():
        _line = _line.strip()
        if not _line or _line.startswith('#') or '=' not in _line:
            continue
        _key, _val = _line.split('=', 1)
        os.environ.setdefault(_key.strip(), _val.strip().strip('"\''))

_hf_token = os.environ.get('HF_TOKEN')
if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token, add_to_git_credential=False)
    print('Hugging Face: авторизация OK')
else:
    print('HF_TOKEN не найден в .env — веса timm скачаются без токена (медленнее)')


In [ ]:
import copy
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from dataset import (
    BuildingDataset,
    get_building_train_transforms,
    get_transforms,
    load_split,
)
from model_convnext import build_convnext_tiny, unfreeze_all, unfreeze_stages, count_trainable_params
from utils import (
    NY_BUILDING as CFG,
    NY_BUILDING_CLASSES as CLASSES,
    IMAGE_SIZE,
    MODELS_DIR,
    NUM_WORKERS,
    PLOT_DPI,
    RANDOM_SEED,
    REPORTS_DIR,
    NY_BUILDING_SPLIT_CSV,
    ensure_dirs,
    set_random_seed,
)

BATCH_SIZE = CFG.batch_size
MAX_EPOCHS = CFG.max_epochs
PATIENCE = CFG.patience
MIN_EPOCHS = CFG.min_epochs
WEIGHT_DECAY = CFG.weight_decay
LABEL_SMOOTHING = CFG.label_smoothing
FOCAL_GAMMA = CFG.focal_gamma

LR_CANDIDATES = CFG.lr_candidates
STAGE1_MAX_EPOCHS = CFG.stage1_max_epochs
STAGE1_MIN_EPOCHS = CFG.stage1_min_epochs
STAGE1_PATIENCE = CFG.stage1_patience
STAGE2_STEPS = CFG.stage2_steps
STAGE2_MIN_EPOCHS_FLOOR = CFG.stage2_min_epochs_floor
STAGE2_MAX_EPOCHS_PER_STEP = CFG.stage2_max_epochs_per_step

ensure_dirs()
set_random_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Устройство:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print(f'Focal gamma={FOCAL_GAMMA}, label_smoothing={LABEL_SMOOTHING}, weight_decay={WEIGHT_DECAY}')
print(f'stage2_max_epochs_per_step={STAGE2_MAX_EPOCHS_PER_STEP}')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = CFG.figure_size


## Загрузка split


In [ ]:
if not NY_BUILDING_SPLIT_CSV.exists():
    raise FileNotFoundError(
        f'{NY_BUILDING_SPLIT_CSV} не найден. Сначала: python scripts/prepare_ny_building_dataset.py && python scripts/build_ny_building_split.py'
    )

split_df = load_split(NY_BUILDING_SPLIT_CSV)
train_df = split_df[split_df['split'] == 'train']

print('Размер по split:')
_split_ru = {'train': 'обучение', 'val': 'валидация', 'test': 'тест'}
_split_counts = split_df['split'].value_counts().reindex(['train', 'val', 'test'])
_split_counts.index = [_split_ru[k] for k in _split_counts.index]
print(_split_counts)
print()
print('Дисбаланс классов (весь набор):')
print(split_df['class'].value_counts())
print()
print('Дисбаланс классов (train):')
print(train_df['class'].value_counts())
print()
print('Напоминание: residential ~2× больше commercial/industrial — WeightedRandomSampler выравнивает батчи.')
print('test = 15% holdout (stratified), не использовался при подборе эпох.')


## Датасеты и лоадеры

`WeightedRandomSampler` для train — каждый класс равновероятен в батче
(без него residential доминировал бы почти во всех батчах).


In [ ]:
eval_transform = get_transforms(image_size=IMAGE_SIZE)
train_transform = get_building_train_transforms(image_size=IMAGE_SIZE, augment=True)

train_ds = BuildingDataset(split_df, split='train', classes=CLASSES, transform=train_transform, use_mask=True)
val_ds = BuildingDataset(split_df, split='val', classes=CLASSES, transform=eval_transform, use_mask=True)
test_ds = BuildingDataset(split_df, split='test', classes=CLASSES, transform=eval_transform, use_mask=True)


def make_weighted_sampler(df, classes=CLASSES) -> WeightedRandomSampler:
    """Каждый класс равновероятен в батче (компенсирует дисбаланс residential ~2×)."""
    class_counts = df['class'].value_counts()
    class_weight = {cls: 1.0 / class_counts[cls] for cls in classes}
    sample_weights = df['class'].map(class_weight).to_numpy()
    return WeightedRandomSampler(sample_weights, num_samples=len(df), replacement=True)


LOADER_WORKERS = 4
LOADER_KWARGS = dict(num_workers=LOADER_WORKERS, persistent_workers=LOADER_WORKERS > 0, pin_memory=True)

train_sampler = make_weighted_sampler(train_df, CLASSES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=train_sampler, **LOADER_KWARGS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **LOADER_KWARGS)

print(f'Обучение: {len(train_ds)} | Валидация: {len(val_ds)} | Тест: {len(test_ds)}')
print('Mask-guided фон + усиленные аугментации включены.')


## Focal loss + вспомогательные функции обучения


In [ ]:
class FocalLoss(nn.Module):
    """Focal loss с label smoothing (для дисбаланса industrial/commercial)."""

    def __init__(self, gamma: float = 2.0, label_smoothing: float = 0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        n_classes = logits.size(1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.label_smoothing / (n_classes - 1) if n_classes > 1 else 0.0)
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)
        pt = (true_dist * log_probs.exp()).sum(dim=1).clamp(min=1e-8)
        focal = (1.0 - pt) ** self.gamma
        loss = -(focal * (true_dist * log_probs).sum(dim=1))
        return loss.mean()


criterion = FocalLoss(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
print(f'FocalLoss gamma={FOCAL_GAMMA}, label_smoothing={LABEL_SMOOTHING}')


def run_epoch(loader, model, criterion, optimizer=None):
    """Один проход: обучение или оценка."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
    return avg_loss, accuracy, macro_f1


@torch.no_grad()
def collect_predictions(loader, model):
    """Собирает предсказания на выборке."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.tolist())
        all_probs.extend(probs.cpu().tolist())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


def train_loop(model, train_loader, val_loader, lr, criterion, max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS,
               patience=PATIENCE, weight_decay=WEIGHT_DECAY, label='', verbose=True):
    """Цикл обучения с early stop по отсутствию роста val macro F1."""
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc', 'val_f1', 'lr']}
    best_val_f1, best_epoch = -1.0, 0
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improve = 0
    prefix = f'[{label}] ' if label else ''

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss, train_acc, _ = run_epoch(train_loader, model, criterion, optimizer)
        val_loss, val_acc, val_f1 = run_epoch(val_loader, model, criterion, optimizer=None)
        scheduler.step(val_f1)
        lr_now = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(lr_now)

        is_best = val_f1 > best_val_f1
        if is_best:
            best_val_f1 = val_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        if verbose:
            marker = ' <- лучшая' if is_best else ''
            print(
                f'{prefix}эпоха {epoch:3d}/{max_epochs} | '
                f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
                f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} macro F1={val_f1:.4f} | '
                f'{time.time() - t0:.1f}s{marker}'
            )

        if epoch >= min_epochs and epochs_without_improve >= patience:
            if verbose:
                print(f'{prefix}early stop: val macro F1 не растёт {patience} эпох')
            break

    return {
        'history': history,
        'best_f1': best_val_f1,
        'best_acc': history['val_acc'][best_epoch - 1],
        'best_epoch': best_epoch,
        'best_state': best_state,
        'epochs_run': len(history['train_loss']),
    }


def print_classification_report_ru(y_true, y_pred, target_names):
    """Отчёт по классам."""
    report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)
    print(f"{'Класс':<22} {'Точн.':>8} {'Полнота':>8} {'F1':>8} {'Объектов':>8}")
    for name in target_names:
        r = report[name]
        print(f"{name:<22} {r['precision']:8.2f} {r['recall']:8.2f} {r['f1-score']:8.2f} {int(r['support']):8d}")
    n = int(report['macro avg']['support'])
    print(f"{'Точность':<22} {report['accuracy']:8.2f} {'':>8} {'':>8} {n:8d}")
    m = report['macro avg']
    print(f"{'Среднее macro':<22} {m['precision']:8.2f} {m['recall']:8.2f} {m['f1-score']:8.2f} {n:8d}")
    w = report['weighted avg']
    print(f"{'Среднее взвеш.':<22} {w['precision']:8.2f} {w['recall']:8.2f} {w['f1-score']:8.2f} {n:8d}")


## Этап 1: подбор learning rate (backbone заморожен, обучается только head)


In [ ]:
import json

STAGE1_CKPT = MODELS_DIR / 'ny_building_stage1_best.pth'
STAGE1_META = MODELS_DIR / 'ny_building_stage1_meta.json'
STAGE2_SUB_CKPT = {i: MODELS_DIR / f'ny_building_stage2_sub{i}_best.pth' for i in (1, 2)}
STAGE2_META = MODELS_DIR / 'ny_building_stage2_meta.json'

SKIP_STAGE1_IF_CHECKPOINT = True  # True = пропустить этап 1, если есть чекпоинт

lr_search_results = {}
stage1_best_state = None
stage1_best_f1 = -1.0
best_lr = None

if SKIP_STAGE1_IF_CHECKPOINT and STAGE1_CKPT.exists() and STAGE1_META.exists():
    meta = json.loads(STAGE1_META.read_text(encoding='utf-8'))
    best_lr = meta['best_lr']
    stage1_best_f1 = meta['best_f1']
    stage1_best_state = torch.load(STAGE1_CKPT, map_location='cpu', weights_only=True)
    print(f'Загружен чекпоинт этапа 1: LR={best_lr:.0e}, macro F1={stage1_best_f1:.4f}')
else:
    for lr in LR_CANDIDATES:
        torch.manual_seed(RANDOM_SEED)
        model = build_convnext_tiny(num_classes=len(CLASSES), freeze_backbone=True).to(device)
        print(f'\n--- LR={lr:.0e} | обучаемых параметров: {count_trainable_params(model):,} ---')

        result = train_loop(
            model, train_loader, val_loader, lr, criterion,
            max_epochs=STAGE1_MAX_EPOCHS,
            min_epochs=STAGE1_MIN_EPOCHS,
            patience=STAGE1_PATIENCE,
            label=f'этап1_{lr:.0e}',
        )

        lr_search_results[lr] = {
            'best_val_f1': result['best_f1'],
            'best_val_acc': result['best_acc'],
            'epochs_run': result['epochs_run'],
            'history': result['history'],
        }

        if result['best_f1'] > stage1_best_f1:
            stage1_best_f1 = result['best_f1']
            stage1_best_state = result['best_state']
            best_lr = lr

    torch.save(stage1_best_state, STAGE1_CKPT)
    STAGE1_META.write_text(
        json.dumps({'best_lr': best_lr, 'best_f1': stage1_best_f1}, indent=2),
        encoding='utf-8',
    )
    print(f'\nЧекпоинт этапа 1 сохранён: {STAGE1_CKPT}')

print(f'\nЛучший LR для этапа 1: {best_lr:.0e} (macro F1={stage1_best_f1:.4f})')


In [ ]:
lr_summary = pd.DataFrame([
    {
        'LR': lr,
        'macro F1': res['best_val_f1'],
        'accuracy': res['best_val_acc'],
        'эпох': res['epochs_run'],
    }
    for lr, res in lr_search_results.items()
])
print(lr_summary)


## Этап 2: прогрессивная разморозка

Чекпоинты: `models/ny_building_stage2_sub1_best.pth`, `..._sub2_best.pth`.

Если обучение прервалось — в следующей ячейке задайте `RESUME_STAGE2_FROM = 2` (продолжить с подэтапа 2).


In [ ]:
import json

STAGE2_SUB_CKPT = {i: MODELS_DIR / f'ny_building_stage2_sub{i}_best.pth' for i in (1, 2)}
STAGE2_META = MODELS_DIR / 'ny_building_stage2_meta.json'

RESUME_STAGE2_FROM = 1  # 1 = с начала | 2 = только подэтап 2 (если sub1 уже сохранён)


def run_progressive_finetune(train_loader, resume_from_substage: int = 1):
    """Этап 2: лимит эпох на подэтап + чекпоинты после каждого подэтапа."""
    substage_results = []
    best_substage = None
    combined_history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc', 'val_f1', 'lr']}
    best_f1, best_acc, best_state = -1.0, 0.0, None
    remaining_epochs = MAX_EPOCHS
    carry_state = stage1_best_state
    carry_lr = None

    if resume_from_substage > 1 and STAGE2_META.exists():
        meta = json.loads(STAGE2_META.read_text(encoding='utf-8'))
        combined_history = meta['history']
        substage_results = meta['substage_results']
        best_f1 = meta['best_f1']
        best_acc = meta['best_acc']
        best_state = torch.load(STAGE2_SUB_CKPT[meta['best_substage']], map_location='cpu', weights_only=True)
        best_substage = meta['best_substage']
        remaining_epochs = MAX_EPOCHS - len(combined_history['train_loss'])
        prev_ckpt = STAGE2_SUB_CKPT[resume_from_substage - 1]
        if prev_ckpt.exists():
            carry_state = torch.load(prev_ckpt, map_location='cpu', weights_only=True)
            carry_lr = meta.get('carry_lr')
            print(f'Возобновление с подэтапа {resume_from_substage}, осталось эпох: {remaining_epochs}')
        else:
            print(f'Чекпоинт {prev_ckpt} не найден — старт с подэтапа 1')
            resume_from_substage = 1
            combined_history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc', 'val_f1', 'lr']}
            substage_results = []
            best_f1, best_acc, best_state = -1.0, 0.0, None
            remaining_epochs = MAX_EPOCHS
            carry_state = stage1_best_state
            carry_lr = None

    for step_idx, (n_stages, lr_divisor) in enumerate(STAGE2_STEPS):
        substage_no = step_idx + 1
        if substage_no < resume_from_substage:
            continue
        if remaining_epochs <= 0:
            break

        torch.manual_seed(RANDOM_SEED)
        model = build_convnext_tiny(num_classes=len(CLASSES), freeze_backbone=True).to(device)
        model.load_state_dict(carry_state)

        if step_idx == len(STAGE2_STEPS) - 1:
            unfreeze_all(model)
            stage_label = 'все слои'
        else:
            unfreeze_stages(model, n_stages)
            stage_label = f'последние {n_stages} стадии'

        target_lr = best_lr / lr_divisor
        fine_tune_lr = min(carry_lr, target_lr) if carry_lr is not None else target_lr
        step_max = min(STAGE2_MAX_EPOCHS_PER_STEP, remaining_epochs)
        print(
            f'\n--- Подэтап {substage_no}/{len(STAGE2_STEPS)}: {stage_label} | '
            f'LR={fine_tune_lr:.1e} | max_epochs={step_max} | '
            f'обучаемых параметров: {count_trainable_params(model):,} ---'
        )

        result = train_loop(
            model, train_loader, val_loader, fine_tune_lr, criterion,
            max_epochs=step_max,
            min_epochs=max(STAGE2_MIN_EPOCHS_FLOOR, MIN_EPOCHS // 3),
            label=f'этап2_{substage_no}',
        )

        for key in combined_history:
            combined_history[key].extend(result['history'][key])

        substage_results.append({
            'substage': substage_no,
            'epochs_run': result['epochs_run'],
            'best_f1': result['best_f1'],
            'best_epoch': result['best_epoch'],
        })

        torch.save(result['best_state'], STAGE2_SUB_CKPT[substage_no])
        print(f'Чекпоинт подэтапа {substage_no} сохранён: {STAGE2_SUB_CKPT[substage_no]}')

        remaining_epochs = MAX_EPOCHS - len(combined_history['train_loss'])
        carry_state = result['best_state']
        carry_lr = result['history']['lr'][-1]

        if result['best_f1'] > best_f1:
            best_f1 = result['best_f1']
            best_acc = result['best_acc']
            best_state = result['best_state']
            best_substage = substage_no

        STAGE2_META.write_text(
            json.dumps({
                'history': combined_history,
                'substage_results': substage_results,
                'best_f1': best_f1,
                'best_acc': best_acc,
                'best_substage': best_substage,
                'carry_lr': carry_lr,
            }, indent=2),
            encoding='utf-8',
        )

    return {
        'history': combined_history,
        'best_f1': best_f1,
        'best_acc': best_acc,
        'best_state': best_state,
        'epochs_run': len(combined_history['train_loss']),
        'substage_results': substage_results,
        'best_substage': best_substage,
    }


stage2_result = run_progressive_finetune(train_loader, resume_from_substage=RESUME_STAGE2_FROM)
print(f'\nЭтап 2 завершён за {stage2_result["epochs_run"]} эпох')
print(f'Лучший macro F1 на валидации: {stage2_result["best_f1"]:.4f}')
print('Подэтапы:', stage2_result['substage_results'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, len(stage2_result['history']['val_loss']) + 1)

axes[0].plot(epochs_range, stage2_result['history']['train_loss'], label='обучение', alpha=0.8)
axes[0].plot(epochs_range, stage2_result['history']['val_loss'], label='валидация', alpha=0.8)
axes[0].set_title('Loss (этап 2)')
axes[0].set_xlabel('эпоха')
axes[0].set_ylabel('loss')
axes[0].legend()

axes[1].plot(epochs_range, stage2_result['history']['val_f1'], label='macro F1', color='green', marker='o', markersize=3)
axes[1].set_title('macro F1 (этап 2)')
axes[1].set_xlabel('эпоха')
axes[1].set_ylabel('macro F1')
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / CFG.training_curves_plot, dpi=PLOT_DPI)
plt.show()


## Сохранение модели


In [ ]:
best_model_path = MODELS_DIR / CFG.model_filename
torch.save(stage2_result['best_state'], best_model_path)

print('ConvNeXt-Tiny, building-классификатор (Zenodo NY small)')
print(f'Accuracy на валидации: {stage2_result["best_acc"]:.4f}')
print(f'macro F1 на валидации: {stage2_result["best_f1"]:.4f}')
print(f'Модель сохранена: {best_model_path}')


## Оценка на валидации


In [ ]:
final_model = build_convnext_tiny(num_classes=len(CLASSES), freeze_backbone=False).to(device)
final_model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
final_model.eval()

val_preds, val_labels, val_probs = collect_predictions(val_loader, final_model)

val_accuracy = float(np.mean(val_preds == val_labels))
val_macro_f1 = float(f1_score(val_labels, val_preds, average='macro', zero_division=0))

print(f'Accuracy на валидации: {val_accuracy:.4f}')
print(f'macro F1 на валидации: {val_macro_f1:.4f}')
print()
print('Отчёт по классам:')
print_classification_report_ru(val_labels, val_preds, CLASSES)


In [ ]:
cm = confusion_matrix(val_labels, val_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
ax.set_xlabel('предсказание')
ax.set_ylabel('истина')
ax.set_title('Confusion matrix (val)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / CFG.confusion_matrix_plot, dpi=PLOT_DPI)
plt.show()


## Примеры предсказаний


In [ ]:
from PIL import Image

val_df = split_df[split_df['split'] == 'val'].reset_index(drop=True)


def show_prediction_examples(paths_df, labels, preds, probs, n_correct_per_class=CFG.n_correct_per_class, seed=RANDOM_SEED):
    """Отображает верные и ошибочные предсказания."""
    rng = np.random.default_rng(seed)

    correct_indices = []
    for cls_idx, cls_name in enumerate(CLASSES):
        cls_correct = np.where((labels == cls_idx) & (preds == cls_idx))[0]
        if len(cls_correct) == 0:
            continue
        n_pick = min(n_correct_per_class, len(cls_correct))
        pick = rng.choice(cls_correct, size=n_pick, replace=False)
        correct_indices.extend(pick.tolist())

    error_indices = np.where(preds != labels)[0].tolist()
    if len(error_indices) > 12:
        error_indices = rng.choice(error_indices, size=12, replace=False).tolist()

    def _plot_grid(indices, title, filename, cols=4):
        if not indices:
            print(f'{title}: нет примеров')
            return
        cols = min(cols, len(indices))
        rows = int(np.ceil(len(indices) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
        axes = np.atleast_1d(axes).ravel()
        for ax in axes[len(indices):]:
            ax.axis('off')
        for i, idx in enumerate(indices):
            img = Image.open(paths_df.iloc[idx]['path']).convert('RGB')
            true_cls = CLASSES[labels[idx]]
            pred_cls = CLASSES[preds[idx]]
            conf = float(probs[idx][preds[idx]])
            axes[i].imshow(img)
            caption = 'истина: ' + true_cls + '\nпредсказание: ' + pred_cls + f'\nуверенность: {conf:.2f}'
            axes[i].set_title(caption, fontsize=9)
            axes[i].axis('off')
        fig.suptitle(title, fontsize=12)
        plt.tight_layout()
        plt.savefig(REPORTS_DIR / filename, dpi=PLOT_DPI, bbox_inches='tight')
        plt.show()

    print(f'Правильных примеров: {len(correct_indices)} (до {n_correct_per_class} на класс)')
    _plot_grid(
        correct_indices,
        'Правильные предсказания (val)',
        CFG.correct_examples_plot,
    )

    print(f'Ошибочных примеров: {len(error_indices)}')
    _plot_grid(
        error_indices,
        'Ошибочные предсказания (val)',
        CFG.error_examples_plot,
        cols=4,
    )


show_prediction_examples(val_df, val_labels, val_preds, val_probs)


## Проверка на тесте (15% holdout, stratified)

`test` не участвовал в подборе LR и early stop — честная оценка обобщения.


In [ ]:
test_preds, test_labels, test_probs = collect_predictions(test_loader, final_model)

test_accuracy = float(np.mean(test_preds == test_labels))
test_macro_f1 = float(f1_score(test_labels, test_preds, average='macro', zero_division=0))

print(f'Accuracy на тесте: {test_accuracy:.4f}')
print(f'macro F1 на тесте: {test_macro_f1:.4f}')
print(f'Разница accuracy (val - test): {val_accuracy - test_accuracy:+.4f}')
print()
print('Отчёт по классам (test):')
print_classification_report_ru(test_labels, test_preds, CLASSES)


## Итоги


In [ ]:
report_dict = classification_report(
    test_labels, test_preds, target_names=CLASSES, output_dict=True, zero_division=0
)
worst_class = min(CLASSES, key=lambda c: report_dict[c]['f1-score'])
best_class = max(CLASSES, key=lambda c: report_dict[c]['f1-score'])

history = stage2_result['history']
best_epoch_idx = int(np.argmax(history['val_f1']))
best_global_epoch = best_epoch_idx + 1
train_acc_at_best = history['train_acc'][best_epoch_idx]
val_acc_at_best = history['val_acc'][best_epoch_idx]
acc_gap = train_acc_at_best - val_acc_at_best
n_errors = int(np.sum(test_preds != test_labels))

print('=' * 60)
print('Итоги: building-классификатор ConvNeXt-Tiny (Zenodo NY small)')
print('=' * 60)
print(f'Выборки: обучение={len(train_ds)} валидация={len(val_ds)} тест={len(test_ds)}')
print('(test = 15% holdout, stratified)')
print()
print('Этап 1 — подбор LR:')
for lr, res in lr_search_results.items():
    marker = ' <- лучший' if lr == best_lr else ''
    print(f'  LR={lr:.0e}: macro F1={res["best_val_f1"]:.4f}, эпох={res["epochs_run"]}{marker}')
print()
print('Этап 2 — разморозка:')
for sub in stage2_result['substage_results']:
    marker = ' <- лучший' if sub['substage'] == stage2_result['best_substage'] else ''
    print(
        f'  Подэтап {sub["substage"]}: {sub["epochs_run"]} эпох, '
        f'лучший macro F1={sub["best_f1"]:.4f} (эпоха {sub["best_epoch"]}){marker}'
    )
print(f'  Всего эпох: {stage2_result["epochs_run"]}/{MAX_EPOCHS}')
print()
print('Финальные метрики:')
print(f'  Лучшая эпоха: {best_global_epoch} (подэтап {stage2_result["best_substage"]})')
print(f'  Валидация:  macro F1={val_macro_f1:.4f}, accuracy={val_accuracy:.4f}')
print(f'  Тест:       macro F1={test_macro_f1:.4f}, accuracy={test_accuracy:.4f}, ошибок={n_errors}')
print(f'  Train acc={train_acc_at_best:.4f}, val acc={val_acc_at_best:.4f}, разрыв={acc_gap:.4f}')
print()
print('По классам (тест):')
print(f'  Лучший  — {best_class}: F1={report_dict[best_class]["f1-score"]:.4f}')
print(f'  Худший  — {worst_class}: F1={report_dict[worst_class]["f1-score"]:.4f}, '
      f'precision={report_dict[worst_class]["precision"]:.4f}, recall={report_dict[worst_class]["recall"]:.4f}')
print()
print(f'Модель сохранена: {best_model_path}')
print('Используется в notebooks/08_zone_building_pipeline.ipynb вместо GT-fallback.')


## Результаты обучения (из сохранённых артефактов)

Если ячейки выше без outputs — обучение уже было: модель и графики лежат в `models/` и `reports/`.
Запустите **только эту секцию** (без переобучения).


In [ ]:
from IPython.display import Image, display
import json

meta1 = json.loads((MODELS_DIR / 'ny_building_stage1_meta.json').read_text(encoding='utf-8'))
meta2 = json.loads((MODELS_DIR / 'ny_building_stage2_meta.json').read_text(encoding='utf-8'))
ckpt = MODELS_DIR / CFG.model_filename

print('Чекпоинт:', ckpt, '| exists:', ckpt.exists())
print('Stage1: LR=', meta1['best_lr'], 'val macro F1=', round(meta1['best_f1'], 4))
print('Stage2: best val macro F1=', round(meta2['best_f1'], 4), '| best substage=', meta2['best_substage'])
print('Best val acc:', round(meta2['best_acc'], 4))

figs = [
    REPORTS_DIR / CFG.training_curves_plot,
    REPORTS_DIR / CFG.confusion_matrix_plot,
    REPORTS_DIR / CFG.correct_examples_plot,
    REPORTS_DIR / CFG.error_examples_plot,
]
for fig in figs:
    if fig.exists():
        print(fig.name)
        display(Image(filename=str(fig)))
    else:
        print('Нет файла:', fig)


In [ ]:
# кривые из meta (если PNG нет)
import matplotlib.pyplot as plt

h = meta2.get('history', {})
if h.get('val_f1'):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h['train_loss'], label='train')
    ax[0].plot(h['val_loss'], label='val')
    ax[0].set_title('Loss'); ax[0].legend()
    ax[1].plot(h['val_f1'], label='val macro F1', color='green')
    ax[1].set_title('Val macro F1'); ax[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print('history пуст')
